# Test — Day 5 Business Rules & Delta ACID

In [0]:
# Runs after 10_silver_business_rules and 12_delta_acid_timetravel_demo.

import unittest
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "silver", "2. Silver Schema")
CATALOG = dbutils.widgets.get("catalog_name")
SILVER = dbutils.widgets.get("silver_schema")
TABLE = f"{CATALOG}.{SILVER}.streets_business"


class BusinessRulesAndAcidTests(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.df = spark.table(TABLE)
        cls.delta_table = DeltaTable.forName(spark, TABLE)

    def test_table_has_rows(self):
        self.assertGreater(self.df.count(), 0, f"{TABLE} is empty.")

    def test_raining_clipped_is_always_in_range(self):
        out_of_range = self.df.filter("raining_clipped < 0 OR raining_clipped > 100").count()
        self.assertEqual(out_of_range, 0, f"{out_of_range} rows still outside [0,100] after clipping.")

    def test_original_raining_preserved(self):
        """The clip rule must not have overwritten the original value."""
        still_has_out_of_range_raw = self.df.filter("raining < 0 OR raining > 100").count()
        self.assertGreater(still_has_out_of_range_raw, 0,
                            "Expected some out-of-range values in the original 'raining' column — "
                            "if none exist, the clip may have overwritten it instead of adding a new column.")

    def test_reading_date_and_hour_populated(self):
        missing = self.df.filter("reading_date IS NULL OR reading_hour IS NULL").count()
        self.assertEqual(missing, 0, f"{missing} rows missing reading_date/reading_hour.")

    def test_table_has_more_than_one_version(self):
        """Confirms the ACID demo (12_delta_acid_timetravel_demo) actually ran an UPDATE."""
        version_count = self.delta_table.history().count()
        self.assertGreater(version_count, 1,
                            f"Only {version_count} version(s) found — run "
                            f"12_delta_acid_timetravel_demo.py before this test.")

    def test_demo_update_present_in_history(self):
        """
        Was spark.conf-based userMetadata tagging — removed, since that
        config isn't available on serverless compute (CONFIG_NOT_AVAILABLE).
        Checks for an UPDATE operation type instead, which needs no config.
        """
        update_ops = self.delta_table.history().filter("operation = 'UPDATE'").count()
        self.assertGreater(update_ops, 0, "No UPDATE operation found in history — demo notebook hasn't run yet.")

    def test_time_travel_query_returns_different_state(self):
        """
        VERSION AS OF 0 (pre-update) should differ from current for the demo
        street. Assumes the intended run sequence: 11_silver_business_rules
        created this table once (version 0), then 12_delta_acid_timetravel_demo
        applied its UPDATE once (version 1+). If 11 was rerun (mode=overwrite)
        AFTER 12's demo UPDATE, version 0 no longer means "before the demo" —
        rerun 12 after any rerun of 11, not before, to keep this assumption true.
        """
        current_avg = spark.sql(
            f"SELECT AVG(pollution) v FROM {TABLE} WHERE street_id = 1"
        ).collect()[0]["v"]
        original_avg = spark.sql(
            f"SELECT AVG(pollution) v FROM {TABLE} VERSION AS OF 0 WHERE street_id = 1"
        ).collect()[0]["v"]
        self.assertNotAlmostEqual(
            current_avg, original_avg, places=6,
            msg=f"Expected version 0 (pre-update) to differ from current after the demo "
                f"correction, got current={current_avg}, version_0={original_avg}."
        )


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(BusinessRulesAndAcidTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Day 5 tests FAILED — see output above.")
